# 🤖 Symptom Triage Chatbot
This chatbot takes user input of symptoms and provides triage-level advice using a rule-based system. For demo purposes, it uses basic logic, but you can upgrade to a fine-tuned model.

In [ ]:
!pip install gradio --quiet

In [ ]:

from transformers import pipeline

# Use a medical-specific instruction-tuned model for triage
triage_llm = pipeline("text-generation", model="TheBloke/meditron-7B-GGUF", device=0)  # Replace with an available HF model


In [ ]:

def triage(symptom_description):
    symptom_description = symptom_description.lower()
    if any(x in symptom_description for x in ["chest pain", "shortness of breath", "severe headache"]):
        return "🚨 URGENT: Seek immediate medical attention or call emergency services."
    elif any(x in symptom_description for x in ["fever", "sore throat", "cough", "cold", "flu"]):
        return "🤒 Primary care recommended. You may see your GP or urgent care."
    elif any(x in symptom_description for x in ["rash", "mild pain", "fatigue", "dizziness"]):
        return "🩺 Non-urgent: Schedule a routine appointment with your doctor."
    else:
        return "❓ Unable to classify. Please consult a healthcare provider."


In [ ]:

def run_triage(user_input):
    prompt = (
        "You are a medical triage assistant. Based on the following symptoms, "
        "provide a likely condition and recommend if urgent care, regular doctor, or home treatment is appropriate.\n"
        f"### Symptoms: {user_input}\n### Assessment:"
    )
    output = triage_llm(prompt, max_length=512, do_sample=True)[0]["generated_text"]
    assessment = output.split("### Assessment:")[-1].strip()
    history.append(f"Input: {user_input}\nAssessment: {assessment}")
    return assessment


In [ ]:

import pandas as pd
import PyPDF2
import gradio as gr

def read_symptoms_from_pdf(pdf_file):
    reader = PyPDF2.PdfReader(pdf_file.name)
    text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
    return text

def read_symptoms_from_csv(csv_file):
    df = pd.read_csv(csv_file.name)
    return "\n".join(df[df.columns[0]].astype(str).tolist())


In [ ]:

# Simple SNOMED-like mapping (illustrative)
snomed_map = {
    "fever": "386661006",
    "cough": "49727002",
    "headache": "25064002",
    "chest pain": "29857009"
}

def map_to_snomed(symptom_text):
    mappings = {symptom: snomed_map.get(symptom.lower()) for symptom in symptom_text.split("\n")}
    return {k: v for k, v in mappings.items() if v}


In [ ]:

import speech_recognition as sr

def speech_to_symptoms(audio_file):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = recognizer.record(source)
    return recognizer.recognize_google(audio)


In [ ]:

import gradio as gr

def triage_pipeline(pdf_file=None, csv_file=None, audio_file=None, manual_text=None):
    symptoms_text = ""
    if pdf_file:
        symptoms_text += read_symptoms_from_pdf(pdf_file) + "\n"
    if csv_file:
        symptoms_text += read_symptoms_from_csv(csv_file) + "\n"
    if audio_file:
        symptoms_text += speech_to_symptoms(audio_file) + "\n"
    if manual_text:
        symptoms_text += manual_text + "\n"

    snomed = map_to_snomed(symptoms_text)
    reasoning = med_llm(f"Triage the following symptoms and give clinical reasoning:\n{symptoms_text.strip()}")

    return symptoms_text.strip(), snomed, reasoning

gr.Interface(
    fn=triage_pipeline,
    inputs=[
        gr.File(label="Upload Symptom PDF", file_types=[".pdf"], optional=True),
        gr.File(label="Upload Symptom CSV", file_types=[".csv"], optional=True),
        gr.Audio(source="upload", type="filepath", label="Upload Symptom Audio", optional=True),
        gr.Textbox(label="Manual Symptom Entry", lines=4, optional=True),
    ],
    outputs=[
        gr.Textbox(label="Extracted Symptom Text"),
        gr.Textbox(label="SNOMED Mappings"),
        gr.Textbox(label="AI Triage Output"),
    ],
    title="Symptom Triage Chatbot (Enhanced)",
    description="Upload symptoms via PDF, CSV, or audio—or enter them manually. Get medical LLM-based triage and SNOMED mappings.",
    allow_flagging="never"
).launch()


In [ ]:

from fpdf import FPDF
import os

def export_triage_report(symptom_text, snomed, reasoning, output_path="triage_report.pdf"):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    pdf.cell(200, 10, txt="Symptom Triage Report", ln=True, align='C')
    pdf.ln(10)

    pdf.multi_cell(0, 10, f"Symptoms:\n{symptom_text}")
    pdf.ln(5)
    pdf.multi_cell(0, 10, f"SNOMED Mappings:\n{snomed}")
    pdf.ln(5)
    pdf.multi_cell(0, 10, f"LLM Clinical Reasoning:\n{reasoning}")

    pdf.output(output_path)
    return output_path


In [ ]:

triage_history = []

def record_history(symptom_text, snomed, reasoning):
    triage_history.append({
        "Symptoms": symptom_text,
        "SNOMED": snomed,
        "Reasoning": reasoning
    })

def export_history_to_csv(path="triage_history.csv"):
    import pandas as pd
    pd.DataFrame(triage_history).to_csv(path, index=False)
    return path


In [ ]:

def triage_and_record(pdf_file=None, csv_file=None, audio_file=None, manual_text=None):
    symptom_text, snomed, reasoning = triage_pipeline(pdf_file, csv_file, audio_file, manual_text)
    record_history(symptom_text, snomed, reasoning)
    report_path = export_triage_report(symptom_text, snomed, reasoning)
    return symptom_text, snomed, reasoning, report_path

def export_history():
    return export_history_to_csv()

gr.Interface(
    fn=triage_and_record,
    inputs=[
        gr.File(label="Upload Symptom PDF", file_types=[".pdf"], optional=True),
        gr.File(label="Upload Symptom CSV", file_types=[".csv"], optional=True),
        gr.Audio(source="upload", type="filepath", label="Upload Symptom Audio", optional=True),
        gr.Textbox(label="Manual Symptom Entry", lines=4, optional=True),
    ],
    outputs=[
        gr.Textbox(label="Extracted Symptom Text"),
        gr.Textbox(label="SNOMED Mappings"),
        gr.Textbox(label="AI Triage Output"),
        gr.File(label="Download Triage Report (PDF)")
    ],
    title="Symptom Triage Chatbot (w/ Report Export)",
    description="Upload symptoms or enter them manually. Get LLM-based triage + SNOMED + downloadable report. Use secondary interface to view/export history.",
    allow_flagging="never"
).launch()

gr.Interface(
    fn=export_history,
    inputs=[],
    outputs=gr.File(label="Download History CSV"),
    title="Export Triage History",
    description="Export all past triage results as CSV file."
).launch()
